# 08 — Visualization Dashboard
**Role: Pipeline & Visualization Engineer**

Queries the Elasticsearch index `grab_sentiment` (produced by `07_spark_streaming.ipynb`) and generates:

1. Sentiment distribution (bar + pie)
2. Sentiment trend over time (line chart)
3. Star-score vs predicted sentiment (heatmap)
4. Word count distribution by sentiment
5. Top-word word clouds per sentiment class
6. Batch vs Streaming throughput comparison
7. Reply rate by sentiment

> **Offline fallback:** if Elasticsearch is not running, the notebook automatically falls back to `cleaned_data.csv` so charts still render.

## 1. Install dependencies

In [ ]:
import subprocess, sys

pkgs = ['elasticsearch==8.13.0', 'matplotlib', 'seaborn', 'wordcloud', 'pandas', 'plotly']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '--quiet'])
print('All visualization deps installed.')

## 2. Configuration

In [ ]:
import os

ES_HOST  = os.getenv('ES_HOST', 'localhost')
ES_PORT  = int(os.getenv('ES_PORT', 9200))
ES_INDEX = 'grab_sentiment'
ES_USER  = os.getenv('ES_USER', 'elastic')
ES_PASS  = os.getenv('ES_PASS', 'changeme')

# Fallback CSV (used when ES is unavailable)
FALLBACK_CSV = 'cleaned_data (2).csv'
for _c in ['cleaned_data (2).csv','cleaned_data.csv','../cleaned_data (2).csv','../cleaned_data.csv']:
    if os.path.exists(_c): FALLBACK_CSV = _c; break

OUTPUT_DIR = './charts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SENTIMENT_COLORS = {
    'Positive': '#2ecc71',
    'Neutral':  '#f39c12',
    'Negative': '#e74c3c',
}
print('Config ready.')

## 3. Load data (Elasticsearch → DataFrame)

In [ ]:
import pandas as pd
from elasticsearch import Elasticsearch

USE_ES = False

try:
    es = Elasticsearch(
        f'http://{ES_HOST}:{ES_PORT}',
        basic_auth=(ES_USER, ES_PASS),
        verify_certs=False,
        request_timeout=5
    )
    if es.ping() and es.indices.exists(index=ES_INDEX):
        USE_ES = True
        print(f'Connected to Elasticsearch — index "{ES_INDEX}"')
except Exception as e:
    print(f'Elasticsearch unavailable ({e}). Using fallback CSV.')

if USE_ES:
    # Scroll all docs from the index
    from elasticsearch.helpers import scan
    hits = scan(es, index=ES_INDEX, query={'query': {'match_all': {}}}, size=500)
    records = [h['_source'] for h in hits]
    df = pd.DataFrame(records)
    print(f'Loaded {len(df):,} docs from Elasticsearch.')
else:
    df = pd.read_csv(FALLBACK_CSV)
    # Map numeric labels to strings if needed
    if 'predicted_sentiment' not in df.columns:
        label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
        df['predicted_sentiment'] = df['sentiment_label'].map(label_map)
    print(f'Loaded {len(df):,} rows from fallback CSV.')

# Normalise column names
if 'predicted_sentiment' not in df.columns and 'sentiment_category' in df.columns:
    df['predicted_sentiment'] = df['sentiment_category']

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df.head(3)

## 4. Chart helpers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

def save_fig(name):
    path = os.path.join(OUTPUT_DIR, f'{name}.png')
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print(f'Saved → {path}')

colors = [SENTIMENT_COLORS.get(s, '#95a5a6') for s in ['Positive', 'Neutral', 'Negative']]

## 5. Chart 1 — Sentiment Distribution

In [ ]:
counts = df['predicted_sentiment'].value_counts().reindex(['Positive', 'Neutral', 'Negative'], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(counts.index, counts.values,
            color=[SENTIMENT_COLORS[s] for s in counts.index])
axes[0].set_title('Sentiment Distribution — Bar Chart', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(
    counts.values, labels=counts.index,
    colors=[SENTIMENT_COLORS[s] for s in counts.index],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Sentiment Distribution — Pie Chart', fontsize=14, fontweight='bold')

save_fig('01_sentiment_distribution')

## 6. Chart 2 — Sentiment Trend Over Time

In [ ]:
df_time = df.dropna(subset=['timestamp']).copy()
df_time['month'] = df_time['timestamp'].dt.to_period('M')

trend = (
    df_time.groupby(['month', 'predicted_sentiment'])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
trend.index = trend.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
for sentiment, color in SENTIMENT_COLORS.items():
    if sentiment in trend.columns:
        ax.plot(trend.index, trend[sentiment], marker='o', linewidth=2,
                label=sentiment, color=color)

ax.set_title('Sentiment Trend Over Time (Monthly)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
ax.legend(title='Sentiment')
plt.xticks(rotation=45)
save_fig('02_sentiment_trend')

## 7. Chart 3 — Star Score vs Predicted Sentiment (Heatmap)

In [ ]:
pivot = (
    df.groupby(['score', 'predicted_sentiment'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['Negative', 'Neutral', 'Positive'], fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pivot, annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Review count'}
)
ax.set_title('Star Score vs Predicted Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Sentiment')
ax.set_ylabel('Star Score (1–5)')
save_fig('03_score_vs_sentiment_heatmap')

## 8. Chart 4 — Word Count Distribution by Sentiment

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for sentiment, color in SENTIMENT_COLORS.items():
    subset = df[df['predicted_sentiment'] == sentiment]['word_count'].dropna()
    if len(subset):
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=sentiment, edgecolor='white')

ax.set_title('Word Count Distribution by Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend(title='Sentiment')
ax.set_xlim(0, 80)
save_fig('04_word_count_distribution')

## 9. Chart 5 — Word Clouds per Sentiment

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sentiments = ['Positive', 'Neutral', 'Negative']

for ax, sentiment in zip(axes, sentiments):
    texts = df[df['predicted_sentiment'] == sentiment]['final_clean_text'].dropna()
    corpus = ' '.join(texts.astype(str).tolist())

    if not corpus.strip():
        ax.set_title(f'{sentiment}\n(no data)')
        ax.axis('off')
        continue

    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap='Greens' if sentiment == 'Positive'
                 else ('Oranges' if sentiment == 'Neutral' else 'Reds'),
        max_words=80,
        contour_width=1, contour_color='steelblue'
    ).generate(corpus)

    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment} Reviews', fontsize=13, fontweight='bold',
                 color=SENTIMENT_COLORS[sentiment])
    ax.axis('off')

fig.suptitle('Word Clouds by Sentiment', fontsize=16, fontweight='bold', y=1.02)
save_fig('05_word_clouds')

## 10. Chart 6 — Batch vs Streaming Throughput Comparison

In [ ]:
# These values come from the 07_spark_streaming.ipynb batch comparison cell.
# Update them after running that notebook.
comparison_data = pd.DataFrame({
    'Mode':             ['Batch Processing', 'Spark Streaming'],
    'Throughput (rows/s)': [12000, 800],      # ← update from actual run
    'Avg Latency (s)':     [0.0,   5.0],      # batch: near-0; streaming: trigger interval
    'Use Case':            ['Historical\nReports', 'Real-Time\nDashboards'],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Throughput
bars = axes[0].bar(
    comparison_data['Mode'],
    comparison_data['Throughput (rows/s)'],
    color=['#3498db', '#9b59b6'], edgecolor='white'
)
axes[0].set_title('Throughput: Batch vs Streaming', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Rows per second')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=11)

# Latency
bars2 = axes[1].bar(
    comparison_data['Mode'],
    comparison_data['Avg Latency (s)'],
    color=['#2ecc71', '#e67e22'], edgecolor='white'
)
axes[1].set_title('Latency: Batch vs Streaming', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Seconds')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{bar.get_height():.1f}s', ha='center', fontsize=11)

save_fig('06_batch_vs_streaming')

## 11. Chart 7 — Reply Rate by Sentiment

In [ ]:
if 'has_reply' in df.columns:
    reply_rate = (
        df.groupby('predicted_sentiment')['has_reply']
        .apply(lambda x: (x == True).sum() / len(x) * 100)
        .reindex(['Positive', 'Neutral', 'Negative'], fill_value=0)
        .round(1)
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(
        reply_rate.index, reply_rate.values,
        color=[SENTIMENT_COLORS[s] for s in reply_rate.index],
        edgecolor='white'
    )
    ax.set_title('Reply Rate by Sentiment (% of reviews with a Grab reply)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Reply Rate (%)')
    ax.set_ylim(0, 100)
    for bar, val in zip(bars, reply_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val}%', ha='center', fontsize=11)
    save_fig('07_reply_rate')
else:
    print('has_reply column not found — skipping reply rate chart.')

## 12. Interactive dashboard with Plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Interactive Sentiment Distribution ─────────────────────────
counts_df = counts.reset_index()
counts_df.columns = ['Sentiment', 'Count']

fig_bar = px.bar(
    counts_df, x='Sentiment', y='Count',
    color='Sentiment',
    color_discrete_map=SENTIMENT_COLORS,
    text='Count',
    title='Interactive: Sentiment Distribution'
)
fig_bar.update_traces(textposition='outside')
fig_bar.show()

In [ ]:
# ── Interactive Time Trend ──────────────────────────────────────
trend_reset = trend.reset_index().melt(id_vars='month', var_name='Sentiment', value_name='Count')

fig_trend = px.line(
    trend_reset, x='month', y='Count', color='Sentiment',
    color_discrete_map=SENTIMENT_COLORS,
    markers=True,
    title='Interactive: Monthly Sentiment Trend'
)
fig_trend.show()

In [ ]:
# ── Interactive Star Score vs Sentiment ────────────────────────
fig_heat = px.density_heatmap(
    df.dropna(subset=['score', 'predicted_sentiment']),
    x='predicted_sentiment', y='score',
    color_continuous_scale='YlOrRd',
    title='Interactive: Star Score vs Predicted Sentiment',
    labels={'predicted_sentiment': 'Predicted Sentiment', 'score': 'Star Score'}
)
fig_heat.show()

## 13. Summary — Chart Index

In [ ]:
import os
chart_files = sorted(os.listdir(OUTPUT_DIR))
print(f'Charts saved to "{OUTPUT_DIR}/":\n')
for f in chart_files:
    print(f'  {f}')

---
### Kibana Index Patterns (if using Kibana)

After running the pipeline, open Kibana at `http://localhost:5601` and:

1. **Stack Management → Index Patterns** → Create pattern `grab_sentiment*`
2. Set **time field** to `ingested_at`
3. Go to **Discover** to explore raw documents
4. Go to **Dashboard → Create** and add:
   - Lens: Donut chart on `predicted_sentiment`
   - Lens: Date histogram on `ingested_at` split by `predicted_sentiment`
   - Lens: Heatmap on `score` × `predicted_sentiment`
   - Lens: Metric tiles for total positive/negative/neutral counts

**Alternatively, use Apache Superset** connected to Elasticsearch via the ES plugin.